# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-Abouelkhair/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Task type: Ranking / scoring

I am working on the Refresh / Content Opportunity Scoring lane. The decision I want to support is which content pages should be reviewed first for possible improvement.

The output will be a priority score for each content page. Pages with higher scores will appear higher in the review queue.

This is a ranking/scoring problem because the main goal is not simply to predict a yes/no outcome for every page. The practical decision is to prioritize a limited number of pages for review. A ranked list is therefore more useful than an unranked classification result.

The eventual action is content review: an editor or search/content team can inspect the highest-priority pages first.

In [1]:
K = 50
print(f"Primary ranking metric: Precision@{K}")
print("Decision supported: which content pages should be reviewed first?")

Primary ranking metric: Precision@50
Decision supported: which content pages should be reviewed first?


## 2. Target or proxy

For this foundation notebook, I will use a starter proxy for content decline: whether the observed recent search trend is classified as down.

The proxy is is_declining_proxy = 1 when trend_direction == "down" and 0 otherwise.

This is only a framing proxy for the starter dataset. It is not the final capstone target because the starter dataset is a cross-sectional 90-day snapshot. In the final warehouse analysis, I will define the target from a later time window so that the model uses past information to rank pages according to an observed future outcome.

The target is therefore intended to represent whether a page deserves higher review priority because of observed or subsequently measured search-performance decline.

In [3]:
import pandas as pd
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Seif-Abouelkhair/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)
df = pd.read_csv(DATA_URL)
df["is_declining_proxy"] = (
    df["trend_direction"].eq("down").astype(int)
)
print("Rows:", len(df))
print("Declining proxy rate:", round(df["is_declining_proxy"].mean(), 3))
print("\nProxy counts:")
print(df["is_declining_proxy"].value_counts())

Rows: 30000
Declining proxy rate: 0.542

Proxy counts:
is_declining_proxy
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

Primary metric: Precision@50

The system will rank content pages by priority and the main decision is which 50 pages should be reviewed first. Precision@50 measures the proportion of those top 50 pages that belong to the observed decline target.

A higher Precision@50 means the limited review queue contains more pages that actually meet the decline criterion.

For this project, a useful result is one where the learned ranking performs better than a simple transparent baseline on the same evaluation set. I will not choose a success threshold after seeing the model results; the metric is defined before modeling.

In [4]:
K = 50
print(f"Primary metric: Precision@{K}")
print("Formula: relevant pages in top K / K")
print("Good means: higher Precision@50 than the agreed baseline on the same test data.")

Primary metric: Precision@50
Formula: relevant pages in top K / K
Good means: higher Precision@50 than the agreed baseline on the same test data.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one pseudonymized content page.

Therefore, one row represents one content item and its aggregated search/content-performance measurements. The starter dataset contains page-level features such as content properties, search-volume information, impressions, clicks, sessions, CTR, position, and recent trend information.

For this framing exercise, the decline proxy is added as a separate column so that the expected target structure is visible. The pseudonymous IDs are used only to identify or group records and will not be used as predictive features.

In [5]:
display(
    df[
        [
            "content_id",
            "client_id",
            "content_type",
            "impressions_90d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "trend_direction",
            "is_declining_proxy",
        ]
    ].head(10)
)

print("One row =", "one pseudonymized content page")
print("Number of pages:", df["content_id"].nunique())
print("Number of clients:", df["client_id"].nunique())

,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,trend_direction,is_declining_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,3970,1,0.03,8.5,down,1
6,content_9a34b442b552,client_8722616204,keyword article,20,0,0.00,7.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,1724,1,0.06,21.2,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,32574,29,0.09,46.0,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,1240,2,0.16,4.9,down,1


One row = one pseudonymized content page
Number of pages: 30000
Number of clients: 32


## 5. Why ML beats a fixed rule here

A simple rule could rank pages using one signal, such as recent impression decline. However, content performance depends on multiple interacting signals, including impressions, clicks, CTR, average position, traffic levels, content characteristics, freshness, and search-demand context.

A fixed threshold would require manually choosing many cutoffs and would treat different combinations of signals in the same way. A learned model can combine multiple signals and discover patterns that are difficult to express as a small set of if-statements.

The purpose of ML here is therefore not to replace human content judgment. It is to improve the prioritization step by producing a more consistent ranked review queue. The final output will be treated as decision-support rather than proof of causality or a prediction of any search-engine algorithm.

In [6]:
candidate_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
]
available = [c for c in candidate_columns if c in df.columns]
print("Candidate signals available in the starter data:")
print(available)
print("\nRows and candidate-signal count:")
print(len(df), "rows,", len(available), "candidate signals")

Candidate signals available in the starter data:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update']

Rows and candidate-signal count:
30000 rows, 7 candidate signals


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.